# Analyse des notes de frais

Ce carnet montre comment créer des agents qui utilisent des plugins pour traiter les notes de frais de déplacement à partir d’images de reçus locaux, générer un e-mail de demande de remboursement, et visualiser les données de dépenses à l’aide d’un graphique en secteurs. Les agents choisissent dynamiquement les fonctions en fonction du contexte de la tâche.

Étapes :  
1. L’agent OCR traite l’image du reçu local et extrait les données de frais de déplacement.  
2. L’agent Email génère un e-mail de demande de remboursement.

### Exemple de scénario de frais de déplacement :  
Imaginez que vous êtes un employé en déplacement pour une réunion d’affaires dans une autre ville. Votre entreprise a une politique de remboursement de toutes les dépenses liées raisonnables au déplacement. Voici une répartition des dépenses potentielles de déplacement :  
- Transport :  
Billet d’avion aller-retour de votre ville d’origine à la ville de destination.  
Taxi ou services de covoiturage vers et depuis l’aéroport.  
Transport local dans la ville de destination (comme les transports publics, les voitures de location ou les taxis).

- Hébergement :  
Séjour à l’hôtel pendant trois nuits dans un hôtel d'affaires de milieu de gamme proche du lieu de la réunion.

- Repas :  
Indemnité journalière pour les repas du petit-déjeuner, déjeuner et dîner, basée sur la politique de per diem de l’entreprise.

- Dépenses diverses :  
Frais de stationnement à l’aéroport.  
Frais d’accès Internet à l’hôtel.  
Pourboires ou petits frais de service.

- Documentation :  
Vous soumettez tous les reçus (vols, taxis, hôtel, repas, etc.) ainsi qu’un rapport de frais complété pour le remboursement.


## Importer les bibliothèques requises

Importer les bibliothèques et modules nécessaires pour le carnet.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Définir les modèles de dépenses

 Créez un modèle Pydantic pour les dépenses individuelles et une classe ExpenseFormatter pour convertir une requête utilisateur en données de dépenses structurées.

 Chaque dépense sera représentée dans le format :
 `{'date': '07-Mar-2025', 'description': 'vol vers la destination', 'amount': 675.99, 'category': 'Transport'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Définition des Outils - Génération de l'Email

Créez une fonction d'outil pour générer un e-mail afin de soumettre une demande de remboursement de frais.  
- Cet outil utilise le décorateur `@tool` du Microsoft Agent Framework.  
- Il calcule le montant total des dépenses et formate les détails dans le corps de l'e-mail.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Outil d'extraction des frais de déplacement à partir d'images de reçus

Créez une fonction d'outil pour extraire les frais de déplacement à partir d'images de reçus.
- Cet outil utilise le décorateur `@tool` du Microsoft Agent Framework.
- Il lit l'image du reçu, l'encode en base64, et retourne l'URI de données pour que l'agent puisse l'analyser.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Traitement des dépenses

Définissez les agents et connectez-les dans un flux de travail séquentiel à l’aide de `WorkflowBuilder`.
- L’agent OCR extrait les données de dépenses structurées à partir de l’image du reçu en utilisant l’outil `load_receipt_image`.
- L’agent Email prend les données extraites et génère un email professionnel de déclaration de dépenses en utilisant l’outil `generate_expense_email`.
- `WorkflowBuilder` avec `add_edge` crée un pipeline séquentiel : Agent OCR → Agent Email.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Fonction principale

Construisez le flux de travail séquentiel et exécutez-le pour traiter l’image du reçu et générer l’e-mail de demande de remboursement des frais.


> **Note :** Ce flux de travail transmet actuellement l'image du reçu sous forme de texte base64, ce que la plupart des modèles de chat (y compris gpt-4o) ne traiteront pas comme une image.  
> Il peut également dépasser la fenêtre de contexte du modèle. Il est préférable d’exécuter la reconnaissance optique de caractères (OCR) avec Azure AI Vision (ou un autre outil OCR) et de ne transmettre que le texte extrait, ou de refactoriser pour envoyer l’image en tant que message `image_url`.  
> Si vous souhaitez uniquement éviter les erreurs de contexte, essayez une image de reçu plus petite ou un modèle avec une fenêtre de contexte plus grande.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
